In [1]:
# create degraded probes

import os
import cv2
from tqdm import tqdm

ROOT = "dataset_split"

SPLITS = ["test", "train"]

BLUR_LEVELS = {
    "blur_medium": 7,
    "blur_high": 13,
}

JPEG_LEVELS = {
    "jpeg_medium": 50,
    "jpeg_high": 20,
}


for split in SPLITS:

    probe_root = os.path.join(ROOT, split, "probe")

    degraded_root = os.path.join(ROOT, split, "degraded_probes")

    os.makedirs(degraded_root, exist_ok=True)

    print(f"\nProcessing {split}")

    for identity in tqdm(os.listdir(probe_root)):

        src_dir = os.path.join(probe_root, identity)

        if not os.path.isdir(src_dir):
            continue

        dst_dir = os.path.join(degraded_root, identity)

        os.makedirs(dst_dir, exist_ok=True)

        for image_file in os.listdir(src_dir):

            image_path = os.path.join(src_dir, image_file)

            img = cv2.imread(image_path)

            if img is None:
                continue

            name = os.path.splitext(image_file)[0]

            # medium blur
            blur_med = cv2.GaussianBlur(img, (7, 7), 0)

            cv2.imwrite(
                os.path.join(dst_dir, f"{name}_blur_medium.jpg"),
                blur_med,
            )

            # high blur
            blur_high = cv2.GaussianBlur(img, (13, 13), 0)

            cv2.imwrite(
                os.path.join(dst_dir, f"{name}_blur_high.jpg"),
                blur_high,
            )

            # medium compression
            cv2.imwrite(
                os.path.join(dst_dir, f"{name}_jpeg_medium.jpg"),
                img,
                [cv2.IMWRITE_JPEG_QUALITY, 50],
            )

            # high compression
            cv2.imwrite(
                os.path.join(dst_dir, f"{name}_jpeg_high.jpg"),
                img,
                [cv2.IMWRITE_JPEG_QUALITY, 20],
            )

print("Degradation Complete")


Processing test


100%|██████████| 30/30 [00:00<00:00, 95.65it/s] 



Processing train


100%|██████████| 100/100 [00:01<00:00, 75.60it/s]

Degradation Complete


In [ ]:
import os
import cv2
import pandas as pd

from tqdm import tqdm
from retinaface import RetinaFace

ROOT = "dataset_split"

metadata_rows = []


for split in ["test", "train"]:

    input_root = os.path.join(ROOT, split, "degraded_probes")

    output_root = os.path.join(ROOT, split, "degraded_probes_cropped")

    os.makedirs(output_root, exist_ok=True)

    print(f"\nCropping {split}")

    for identity in tqdm(os.listdir(input_root)):

        src_dir = os.path.join(input_root, identity)

        if not os.path.isdir(src_dir):
            continue

        dst_dir = os.path.join(output_root, identity)

        os.makedirs(dst_dir, exist_ok=True)

        for image_file in os.listdir(src_dir):
            print(f"\nProcessing image: {image_file}")

            image_path = os.path.join(src_dir, image_file)

            try:
                print("Reading image")
                img = cv2.imread(image_path)

                if img is None:
                    continue
                print("iamge loaded")
                print("starting retina face")

                faces = RetinaFace.detect_faces(image_path)
                print("retina face finsihed")
                if not isinstance(faces, dict):
                    print("No faces detected")
                    continue

                largest_area = -1
                best_box = None
                best_score = None

                for face_id in faces:

                    x1, y1, x2, y2 = faces[face_id]["facial_area"]

                    score = faces[face_id]["score"]

                    area = (x2 - x1) * (y2 - y1)

                    if area > largest_area:

                        largest_area = area

                        best_box = (x1, y1, x2, y2)

                        best_score = score

                if best_box is None:
                    continue

                x1, y1, x2, y2 = best_box

                crop = img[y1:y2, x1:x2]
                if crop.size==0:
                    continue
                crop = cv2.resize(crop, (112, 112))

                save_path = os.path.join(dst_dir, image_file)

                cv2.imwrite(save_path, crop)

                metadata_rows.append(
                    {
                        "split": split,
                        "subset": "degraded_probes",
                        "identity": identity,
                        "image_name": image_file,
                        "image_path": os.path.abspath(save_path),
                        "det_score": float(best_score),
                    }
                )
                print(
                    f"{split} | {identity} | {image_file} | Detection Score = {best_score:.6f}"
                )

            except Exception as e:
                print(f"\nERROR: {image_path}")
                print(e)
                continue


metadata_df = pd.DataFrame(metadata_rows)

metadata_df.to_csv("face_detection_scores_degraded.csv", index=False)
print("\nDetection Score Statistics")
print(metadata_df["det_score"].describe())
print("Saved detection scores")


Cropping test


  0%|          | 0/30 [00:00<?, ?it/s]


Processing image: Jennifer_Aniston_0001_jpeg_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
test | Jennifer_Aniston | Jennifer_Aniston_0001_jpeg_high.jpg | Detection Score = 0.998438

Processing image: Jennifer_Aniston_0013_jpeg_medium.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
test | Jennifer_Aniston | Jennifer_Aniston_0013_jpeg_medium.jpg | Detection Score = 0.998743

Processing image: Jennifer_Aniston_0001_blur_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
test | Jennifer_Aniston | Jennifer_Aniston_0001_blur_high.jpg | Detection Score = 0.998203

Processing image: Jennifer_Aniston_0014_jpeg_medium.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
test | Jennifer_Aniston | Jennifer_Aniston_0014_jpeg_medium.jpg | Detection Score = 0.998206

Processing image: Jennifer_Aniston_0008_blur_medium.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
test |

  3%|▎         | 1/30 [00:44<21:42, 44.92s/it]

retina face finsihed
test | Jennifer_Aniston | Jennifer_Aniston_0014_blur_high.jpg | Detection Score = 0.998881

Processing image: Walter_Mondale_0010_jpeg_medium.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
test | Walter_Mondale | Walter_Mondale_0010_jpeg_medium.jpg | Detection Score = 0.999249

Processing image: Walter_Mondale_0007_blur_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
test | Walter_Mondale | Walter_Mondale_0007_blur_high.jpg | Detection Score = 0.999640

Processing image: Walter_Mondale_0010_blur_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
test | Walter_Mondale | Walter_Mondale_0010_blur_high.jpg | Detection Score = 0.999706

Processing image: Walter_Mondale_0010_jpeg_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
test | Walter_Mondale | Walter_Mondale_0010_jpeg_high.jpg | Detection Score = 0.999338

Processing image: Walter_Mondale_0007_jpeg_high

...

retina face finsihed
train | John_Snow | John_Snow_0002_blur_medium.jpg | Detection Score = 0.999740

Processing image: David_Beckham_0011_jpeg_medium.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
train | David_Beckham | David_Beckham_0011_jpeg_medium.jpg | Detection Score = 0.999235

Processing image: David_Beckham_0006_blur_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
train | David_Beckham | David_Beckham_0006_blur_high.jpg | Detection Score = 0.999641

Processing image: David_Beckham_0011_blur_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
train | David_Beckham | David_Beckham_0011_blur_high.jpg | Detection Score = 0.997535

Processing image: David_Beckham_0002_blur_medium.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
train | David_Beckham | David_Beckham_0002_blur_medium.jpg | Detection Score = 0.999323

Processing image: David_Beckham_0002_blur_high.jpg
Reading ima

 57%|█████▋    | 57/100 [27:04<26:43, 37.28s/it]

retina face finsihed
train | David_Beckham | David_Beckham_0011_blur_medium.jpg | Detection Score = 0.998995

Processing image: Donald_Rumsfeld_0060_blur_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
train | Donald_Rumsfeld | Donald_Rumsfeld_0060_blur_high.jpg | Detection Score = 0.999667

Processing image: Donald_Rumsfeld_0033_blur_medium.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
train | Donald_Rumsfeld | Donald_Rumsfeld_0033_blur_medium.jpg | Detection Score = 0.999283

Processing image: Donald_Rumsfeld_0017_jpeg_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
train | Donald_Rumsfeld | Donald_Rumsfeld_0017_jpeg_high.jpg | Detection Score = 0.999222

Processing image: Donald_Rumsfeld_0059_blur_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
train | Donald_Rumsfeld | Donald_Rumsfeld_0059_blur_high.jpg | Detection Score = 0.999545

Processing image: Donald_Rumsfeld_

 58%|█████▊    | 58/100 [31:18<1:11:27, 102.09s/it]

retina face finsihed
train | Donald_Rumsfeld | Donald_Rumsfeld_0118_jpeg_high.jpg | Detection Score = 0.999609

Processing image: Arnold_Schwarzenegger_0013_jpeg_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
train | Arnold_Schwarzenegger | Arnold_Schwarzenegger_0013_jpeg_high.jpg | Detection Score = 0.999560

Processing image: Arnold_Schwarzenegger_0039_jpeg_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
train | Arnold_Schwarzenegger | Arnold_Schwarzenegger_0039_jpeg_high.jpg | Detection Score = 0.999498

Processing image: Arnold_Schwarzenegger_0041_blur_medium.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
train | Arnold_Schwarzenegger | Arnold_Schwarzenegger_0041_blur_medium.jpg | Detection Score = 0.999576

Processing image: Arnold_Schwarzenegger_0007_blur_high.jpg
Reading image
iamge loaded
starting retina face
retina face finsihed
train | Arnold_Schwarzenegger | Arnold_Schwarzenegger_0007_blur

In [ ]:
import os
import cv2
from retinaface import RetinaFace
from tqdm import tqdm

ROOT = "dataset_split"

for split in ["test", "train"]:

    input_root = os.path.join(ROOT, split, "gallery")

    output_root = os.path.join(ROOT, split, "gallery_cropped")

    os.makedirs(output_root, exist_ok=True)

    print(f"\nCropping gallery {split}")

    for identity in tqdm(os.listdir(input_root)):

        src_dir = os.path.join(input_root, identity)

        if not os.path.isdir(src_dir):
            continue

        dst_dir = os.path.join(output_root, identity)

        os.makedirs(dst_dir, exist_ok=True)

        for image_file in os.listdir(src_dir):

            image_path = os.path.join(src_dir, image_file)

            img = cv2.imread(image_path)

            if img is None:
                continue

            faces = RetinaFace.detect_faces(image_path)

            if not isinstance(faces, dict):
                continue

            largest_area = -1
            best_box = None

            for face_id in faces:

                x1, y1, x2, y2 = faces[face_id]["facial_area"]

                area = (x2 - x1) * (y2 - y1)

                if area > largest_area:

                    largest_area = area

                    best_box = (x1, y1, x2, y2)

            if best_box is None:
                continue

            x1, y1, x2, y2 = best_box

            crop = img[y1:y2, x1:x2]

            crop = cv2.resize(crop, (112, 112))

            cv2.imwrite(
                os.path.join(dst_dir, image_file),
                crop,
            )

print("Gallery Cropping Done")